# TUGAS MANDIRI PRAKTIKUM BIG DATA — PERTEMUAN 10
## Pemrosesan Data Streaming dengan Spark Structured Streaming

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 10 (Tugas Mandiri) |
| **Nama Mahasiswa** | Yarsya |
| **NPM** | 2505060066 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |

---

### Skenario Tugas
Manajemen **"Kopi Nusantara"** membangun purwarupa (*prototype*) dashboard pemantauan pesanan **real-time** menggunakan **Spark Structured Streaming**. Pipeline ini membaca aliran berkas transaksi CSV secara inkremental dari direktori simulasi streaming, melakukan agregasi kumulatif per cabang dan berbasis *time window* 2 menit secara paralel di memori, serta menyimpan aliran data mentah secara berkelanjutan ke format **Parquet** (*file sink*).

## 0. Inisialisasi SparkSession & Konfigurasi Shuffle
Menyiapkan `SparkSession` dan mengatur `spark.sql.shuffle.partitions` menjadi `4` agar eksekusi micro-batch pada lingkungan lokal berjalan optimal dan responsif.

In [1]:
import os
import sys
import shutil
import time
from datetime import datetime, timedelta
import pandas as pd

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, window, sum as spark_sum, count

spark = SparkSession.builder \
    .appName("Tugas10-StructuredStreaming-Yarsya") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil diinisialisasi! Versi:", spark.version)

ERROR StatusLogger Could not determine local host name
 java.net.UnknownHostException: sandbox: sandbox: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1670)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1705)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:711)
	at org.apache.logging.log4j.core.LoggerContext.start(LoggerContext.java:253)
	at org.apache.logging.log4j.core.impl.Log4jContextFactory.getContext(Log4jContextFactory.java:245)
	at org.apache.logging.log4j.core.impl.Log4jContextFactory.getContext(Log

ERROR StatusConsoleListener Could not determine local host name
 java.net.UnknownHostException: sandbox: sandbox: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1670)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1705)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.setConfigLocation(LoggerContext.java:679)
	at org.apache.spark.internal.Logging.initializeLogging(Logging.scala:139)
	at org.apache.spark.internal.Logging.initializeLogIfNecessary(Logging.scala:114)
	at org.apache.spark.internal.Logging.initializeLogIfNecessary$(Logging.scala:1

ERROR StatusConsoleListener Could not determine local host name
 java.net.UnknownHostException: sandbox: sandbox: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1670)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1705)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:711)
	at org.apache.spark.internal.Logging$.uninitialize(Logging.scala:220)
	at org.apache.spark.deploy.SparkSubmit.org$apache$spark$deploy$SparkSubmit$$runMain(SparkSubmit.scala:972)
	at org.apache.spark.deploy.SparkSubmit.doRunMain$1(SparkSubmit

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 16:10:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/26 16:10:39 WARN MacAddressUtil: Failed to find a usable hardware address from the network interfaces; using random bytes: ef:8c:90:b7:08:c7:63:e6


SparkSession berhasil diinisialisasi! Versi: 3.5.9


---
## Bagian A. Menyiapkan Streaming Source (Bobot 15%)

Mendefinisikan skema eksplisit (`StructType`) untuk data pesanan (`pesanan_id`, `nama_cabang`, `nama_menu`, `qty`, `harga_satuan`, `waktu_pesan`), membuat folder simulasi `tugas10_stream`, dan mendefinisikan `readStream`.

In [2]:
# Menyiapkan folder simulasi streaming (kosongkan dulu agar clean)
folder_stream = "tugas10_stream"
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

for folder in [folder_stream, folder_output, folder_checkpoint]:
    shutil.rmtree(folder, ignore_errors=True)

os.makedirs(folder_stream, exist_ok=True)

# Mendefinisikan skema eksplisit (wajib untuk file-based streaming)
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

# Mendefinisikan input streaming source (readStream)
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("Skema pesanan berhasil didefinisikan.")
print("df_stream.isStreaming:", df_stream.isStreaming)
df_stream.printSchema()

Skema pesanan berhasil didefinisikan.
df_stream.isStreaming: True
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



---
## Bagian B. Agregasi per Cabang (Bobot 20%)

Menambahkan kolom turunan `total_harga` (`qty * harga_satuan`), kemudian mengelompokkan berdasarkan `nama_cabang` untuk menghitung total omzet dan jumlah pesanan menggunakan `memory` sink bernama `agregasi_cabang`.

In [3]:
# Menambahkan kolom total_harga
df_stream_hitung = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi per cabang
df_agregasi_cabang = df_stream_hitung.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming query pertama ke memory sink
query_cabang = df_agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming Query 1 (agregasi_cabang) dimulai! Aktif:", query_cabang.isActive)

Streaming Query 1 (agregasi_cabang) dimulai! Aktif: True


---
## Bagian C. Agregasi Berbasis Window Waktu (Bobot 20%)

Mendefinisikan ulang `readStream` dari folder `tugas10_stream` yang sama untuk membuat streaming query kedua yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`, dengan sink `memory` bernama `agregasi_waktu`.

In [4]:
# Mendefinisikan ulang readStream dari folder yang sama untuk query kedua
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi berbasis window waktu 2 menit
df_agregasi_waktu = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming query kedua ke memory sink
query_waktu = df_agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming Query 2 (agregasi_waktu) dimulai! Aktif:", query_waktu.isActive)

Streaming Query 2 (agregasi_waktu) dimulai! Aktif: True


---
## Bagian D. Menguji dengan Data Simulasi Multi-Batch (Bobot 25%)

Mengirimkan **3 batch data** CSV secara bertahap dengan jeda `time.sleep()` di antaranya untuk mengamati akumulasi stateful secara *real-time* pada tabel `agregasi_cabang` dan `agregasi_waktu`.

In [5]:
# Menyiapkan timestamp dasar
base_time = datetime(2026, 11, 19, 10, 0, 0)

# === BATCH 1 ===
batch_1 = pd.DataFrame({
    "pesanan_id": ["ORD-01", "ORD-02", "ORD-03"],
    "nama_cabang": ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Magelang"],
    "nama_menu": ["Kopi Susu", "Americano", "Roti Bakar"],
    "qty": [2, 1, 3],
    "harga_satuan": [22000, 20000, 18000],
    "waktu_pesan": [
        (base_time + timedelta(seconds=10)).strftime("%Y-%m-%d %H:%M:%S"),
        (base_time + timedelta(seconds=35)).strftime("%Y-%m-%d %H:%M:%S"),
        (base_time + timedelta(seconds=55)).strftime("%Y-%m-%d %H:%M:%S"),
    ]
})
batch_1.to_csv(f"{folder_stream}/batch_1.csv", index=False)
print("[Batch 1] Terkirim (2 Magelang, 1 Yogyakarta). Menunggu pemrosesan micro-batch...")
time.sleep(8)

print("Hasil agregasi_cabang setelah Batch 1:")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show(truncate=False)

[Batch 1] Terkirim (2 Magelang, 1 Yogyakarta). Menunggu pemrosesan micro-batch...


Hasil agregasi_cabang setelah Batch 1:


+-------------------------+----------------+--------------+
|nama_cabang              |total_pendapatan|jumlah_pesanan|
+-------------------------+----------------+--------------+
|Kopi Nusantara Magelang  |98000           |2             |
|Kopi Nusantara Yogyakarta|20000           |1             |
+-------------------------+----------------+--------------+



In [6]:
# === BATCH 2 ===
batch_2 = pd.DataFrame({
    "pesanan_id": ["ORD-04", "ORD-05", "ORD-06"],
    "nama_cabang": ["Kopi Nusantara Semarang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"],
    "nama_menu": ["Cappuccino", "Kopi Susu", "Kentang Goreng"],
    "qty": [2, 4, 2],
    "harga_satuan": [25000, 22000, 20000],
    "waktu_pesan": [
        (base_time + timedelta(minutes=1, seconds=15)).strftime("%Y-%m-%d %H:%M:%S"),
        (base_time + timedelta(minutes=2, seconds=20)).strftime("%Y-%m-%d %H:%M:%S"),
        (base_time + timedelta(minutes=2, seconds=50)).strftime("%Y-%m-%d %H:%M:%S"),
    ]
})
batch_2.to_csv(f"{folder_stream}/batch_2.csv", index=False)
print("[Batch 2] Terkirim (2 Semarang, 1 Yogyakarta). Menunggu pemrosesan micro-batch...")
time.sleep(6)

print("Hasil agregasi_cabang setelah Batch 2 (Terakumulasi):")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show(truncate=False)

[Batch 2] Terkirim (2 Semarang, 1 Yogyakarta). Menunggu pemrosesan micro-batch...


Hasil agregasi_cabang setelah Batch 2 (Terakumulasi):
+-------------------------+----------------+--------------+
|nama_cabang              |total_pendapatan|jumlah_pesanan|
+-------------------------+----------------+--------------+
|Kopi Nusantara Yogyakarta|108000          |2             |
|Kopi Nusantara Magelang  |98000           |2             |
|Kopi Nusantara Semarang  |90000           |2             |
+-------------------------+----------------+--------------+



In [7]:
# === BATCH 3 ===
batch_3 = pd.DataFrame({
    "pesanan_id": ["ORD-07", "ORD-08", "ORD-09"],
    "nama_cabang": ["Kopi Nusantara Magelang", "Kopi Nusantara Semarang", "Kopi Nusantara Yogyakarta"],
    "nama_menu": ["Cappuccino", "Kopi Susu", "Americano"],
    "qty": [3, 5, 2],
    "harga_satuan": [25000, 22000, 20000],
    "waktu_pesan": [
        (base_time + timedelta(minutes=3, seconds=10)).strftime("%Y-%m-%d %H:%M:%S"),
        (base_time + timedelta(minutes=4, seconds=5)).strftime("%Y-%m-%d %H:%M:%S"),
        (base_time + timedelta(minutes=4, seconds=45)).strftime("%Y-%m-%d %H:%M:%S"),
    ]
})
batch_3.to_csv(f"{folder_stream}/batch_3.csv", index=False)
print("[Batch 3] Terkirim (1 Magelang, 1 Semarang, 1 Yogyakarta). Menunggu pemrosesan micro-batch...")
time.sleep(6)

print("Hasil agregasi_cabang setelah Batch 3 (Total Keseluruhan):")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show(truncate=False)

print("Hasil agregasi_waktu (Window 2 Menit):")
spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_pesanan FROM agregasi_waktu ORDER BY window.start").show(truncate=False)

[Batch 3] Terkirim (1 Magelang, 1 Semarang, 1 Yogyakarta). Menunggu pemrosesan micro-batch...


Hasil agregasi_cabang setelah Batch 3 (Total Keseluruhan):
+-------------------------+----------------+--------------+
|nama_cabang              |total_pendapatan|jumlah_pesanan|
+-------------------------+----------------+--------------+
|Kopi Nusantara Semarang  |200000          |3             |
|Kopi Nusantara Magelang  |173000          |3             |
|Kopi Nusantara Yogyakarta|148000          |3             |
+-------------------------+----------------+--------------+

Hasil agregasi_waktu (Window 2 Menit):
+-------------------+-------------------+----------------+--------------+
|start              |end                |total_pendapatan|jumlah_pesanan|
+-------------------+-------------------+----------------+--------------+
|2026-11-19 10:00:00|2026-11-19 10:02:00|168000          |4             |
|2026-11-19 10:02:00|2026-11-19 10:04:00|203000          |3             |
|2026-11-19 10:04:00|2026-11-19 10:06:00|150000          |2             |
+-------------------+----------------

---
## Bagian E. Sink Berbasis Berkas — Parquet Append Sink (Bobot 20%)

Membangun **streaming query ketiga** yang membaca DataFrame mentah dari `tugas10_stream` dan menuliskannya langsung (mode `append`, tanpa agregasi) ke folder lokal `tugas10_output` dalam format **Parquet** lengkap dengan direktori checkpoint `tugas10_checkpoint`.

Kemudian mengirimkan satu batch data baru (`batch_4.csv`), memeriksa berkas fisik hasil keluaran di `tugas10_output`, dan membacanya kembali sebagai DataFrame batch biasa (`spark.read.parquet()`).

In [8]:
# Streaming query ketiga: menulis data mentah dari df_stream ke Parquet sink
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_checkpoint) \
    .outputMode("append") \
    .start()

print("Streaming Query 3 (Parquet file sink) dimulai! Aktif:", query3.isActive)
time.sleep(6)

# Mengirim satu batch data baru (Batch 4)
batch_4 = pd.DataFrame({
    "pesanan_id": ["ORD-10", "ORD-11"],
    "nama_cabang": ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta"],
    "nama_menu": ["Es Teh", "Cappuccino"],
    "qty": [2, 2],
    "harga_satuan": [15000, 25000],
    "waktu_pesan": [
        (base_time + timedelta(minutes=5, seconds=10)).strftime("%Y-%m-%d %H:%M:%S"),
        (base_time + timedelta(minutes=5, seconds=30)).strftime("%Y-%m-%d %H:%M:%S"),
    ]
})
batch_4.to_csv(f"{folder_stream}/batch_4.csv", index=False)
print("[Batch 4] Terkirim. Menunggu penulisan ke Parquet sink...")
time.sleep(6)

Streaming Query 3 (Parquet file sink) dimulai! Aktif: True


[Batch 4] Terkirim. Menunggu penulisan ke Parquet sink...


In [9]:
# Memeriksa berkas fisik Parquet di folder tugas10_output
!ls -la tugas10_output

# Membaca kembali folder Parquet output sebagai DataFrame batch biasa
df_parquet_batch = spark.read.parquet(folder_output)
print(f"\nTotal baris terbaca dari Parquet sink: {df_parquet_batch.count()} baris")
df_parquet_batch.orderBy("pesanan_id").show(truncate=False)

total 44
drwxr-xr-x 3 meil meil 4096 Nov 26 16:11 .
drwxrwxr-x 5 meil meil 4096 Nov 26 16:11 ..
-rw-r--r-- 1 meil meil 1880 Nov 26 16:11 part-00000-2f8b3586-8c0f-4bf5-b4e1-62587eac9955-c000.snappy.parquet
-rw-r--r-- 1 meil meil   24 Nov 26 16:11 .part-00000-2f8b3586-8c0f-4bf5-b4e1-62587eac9955-c000.snappy.parquet.crc
-rw-r--r-- 1 meil meil 1966 Nov 26 16:11 part-00000-57aefd1c-1658-48a4-ab74-125205cfd815-c000.snappy.parquet
-rw-r--r-- 1 meil meil   24 Nov 26 16:11 .part-00000-57aefd1c-1658-48a4-ab74-125205cfd815-c000.snappy.parquet.crc
-rw-r--r-- 1 meil meil 1929 Nov 26 16:11 part-00001-38cf8d53-9b9b-4605-a43c-2b23857644e9-c000.snappy.parquet
-rw-r--r-- 1 meil meil   24 Nov 26 16:11 .part-00001-38cf8d53-9b9b-4605-a43c-2b23857644e9-c000.snappy.parquet.crc
-rw-r--r-- 1 meil meil 1908 Nov 26 16:11 part-00002-2e7f73f9-b116-4eee-874e-af0025f75cb9-c000.snappy.parquet
-rw-r--r-- 1 meil meil   24 Nov 26 16:11 .part-00002-2e7f73f9-b116-4eee-874e-af0025f75cb9-c000.snappy.parquet.crc
drwxr-xr-x 2


Total baris terbaca dari Parquet sink: 11 baris
+----------+-------------------------+--------------+---+------------+-------------------+
|pesanan_id|nama_cabang              |nama_menu     |qty|harga_satuan|waktu_pesan        |
+----------+-------------------------+--------------+---+------------+-------------------+
|ORD-01    |Kopi Nusantara Magelang  |Kopi Susu     |2  |22000       |2026-11-19 10:00:10|
|ORD-02    |Kopi Nusantara Yogyakarta|Americano     |1  |20000       |2026-11-19 10:00:35|
|ORD-03    |Kopi Nusantara Magelang  |Roti Bakar    |3  |18000       |2026-11-19 10:00:55|
|ORD-04    |Kopi Nusantara Semarang  |Cappuccino    |2  |25000       |2026-11-19 10:01:15|
|ORD-05    |Kopi Nusantara Yogyakarta|Kopi Susu     |4  |22000       |2026-11-19 10:02:20|
|ORD-06    |Kopi Nusantara Semarang  |Kentang Goreng|2  |20000       |2026-11-19 10:02:50|
|ORD-07    |Kopi Nusantara Magelang  |Cappuccino    |3  |25000       |2026-11-19 10:03:10|
|ORD-08    |Kopi Nusantara Semarang  |Kop

---
## Penutupan & Penghentian Seluruh Streaming Query
Menghentikan ketiga streaming query secara eksplisit (`query.stop()`) serta menutup `SparkSession`.

In [10]:
query_cabang.stop()
query_waktu.stop()
query3.stop()

print("Status Query 1 (agregasi_cabang) aktif:", query_cabang.isActive)
print("Status Query 2 (agregasi_waktu)  aktif:", query_waktu.isActive)
print("Status Query 3 (parquet_sink)    aktif:", query3.isActive)

spark.stop()
print("Seluruh streaming query dihentikan dan SparkSession ditutup dengan aman.")

Status Query 1 (agregasi_cabang) aktif: False
Status Query 2 (agregasi_waktu)  aktif: False
Status Query 3 (parquet_sink)    aktif: False


Seluruh streaming query dihentikan dan SparkSession ditutup dengan aman.
